In [1]:
from dotenv import load_dotenv,find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
   raise ValueError('GEMINI_API_KEY is not set in the env file')
print('GEMINI_API_KEY loaded successfully')    
llm = ChatGoogleGenerativeAI(api_key=GEMINI_API_KEY,model='gemini-1.5-flash')

GEMINI_API_KEY loaded successfully


In [2]:
from IPython.display import Markdown
Markdown((llm.invoke('hi').content))

Hi there! How can I help you today?


In [3]:
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph,START,END
from IPython.display import Markdown
class GraphState(TypedDict):
    story_theme: str
    generated_story: str
    
def generate_story(state:GraphState)->GraphState:
    story_theme = state['story_theme']
    generate_story_prompt ="""
    Imagine you are a skilled storyteller tasked with creating a simple and interesting story based on the theme below.  
  
    Theme: {theme}
    Your goal is to generate a detailed and realistic story for 10 years old children
    The story should have:
    - A clear beginning, middle, and end.
    - Easy-to-understand language.
    - Detail and meaningful sentences.
    - Interesting characters and events.
    
    Story:
    """
    

    sys_msg = SystemMessage(content="You are a storyteller who writes clear and engaging stories in simple English.")
    hum_msg =  HumanMessage(content=generate_story_prompt.format(theme=story_theme))
    resp = llm.invoke([sys_msg,hum_msg])
    state['generated_story'] = resp.content
    return state 


workflow = StateGraph(GraphState)
workflow.add_node('generate_story',generate_story)
workflow.add_edge(START,'generate_story')
workflow.add_edge('generate_story',END)
app = workflow.compile()
story_theme =  "Once, a rabbit mocked a slow-moving tortoise. The tortoise challenged him to a race. Confident of his speed, the rabbit dashed ahead and took a nap. Meanwhile, the tortoise kept moving steadily. By the time the rabbit woke up, the tortoise had already crossed the finish line."
resp1 = app.invoke({'story_theme':story_theme})
 
Markdown(resp1['generated_story'])

Barnaby Bunson was a rabbit known for two things: his fluffy white tail and his incredible speed.  He zoomed through Sunny Meadow, a blur of white fur, leaving other animals in his dust. One sunny afternoon, Barnaby hopped past Sheldon the tortoise, who was slowly, slowly making his way towards a juicy dandelion.

"Goodness, Sheldon!" Barnaby chuckled, his nose twitching. "You're slower than a snail in mud!  Why don't you just stay put?"

Sheldon, though slow, had a wise, steady gaze.  "Perhaps," he said calmly, "speed isn't everything, Barnaby.  How about a race? To the big oak tree at the edge of the meadow?"

Barnaby burst into laughter.  "A race? With *you*?  That's the funniest thing I've heard all day! I'll win before you even reach the first blade of grass!"

He agreed to the race, and the other meadow animals gathered to watch.  The starting line was a mossy rock.  Mr. Fox, the race official, shouted, "Ready... Set... GO!"

Barnaby shot off like an arrow, leaving Sheldon in a cloud of dust.  He was so confident, he could practically taste victory.  He raced past the babbling brook, past Mrs. Badger's burrow, and even past the grumpy old owl's tree.  Feeling very smug, Barnaby spotted a patch of sweet clover.  "Just a little nap," he thought, curling up for a quick rest.

Meanwhile, Sheldon, with his slow but steady pace, kept moving. He didn't stop to admire the flowers, or to chat with the butterflies. He just kept putting one tiny foot in front of the other.  He passed the babbling brook, he passed Mrs. Badger's burrow, and he even passed the grumpy old owl's tree.

Barnaby woke with a jolt.  The sun was starting to set.  He looked around frantically.  Where was everyone?  He saw a crowd gathered near the big oak tree.  He raced towards them, his heart pounding.  There, sitting proudly beside the oak tree, was Sheldon, munching on a delicious dandelion.  He had won!

Barnaby was surprised and a little embarrassed.  He learned a valuable lesson that day: steady persistence can beat even the fastest speed.  From that day on, Barnaby was still fast, but he also learned to appreciate the importance of perseverance and not to underestimate anyone, no matter how slow they seemed.  And Sheldon?  Well, Sheldon became a legend in Sunny Meadow, a reminder that slow and steady wins the race.


In [4]:
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
from typing import Sequence
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage
from typing import Annotated
from IPython.display import display, Markdown
import time


class MainCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class SupportingCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class ScenesList(TypedDict):
    id: str
    scene: str
    description: str
    narration: str
    img_prompt: str
    object_description: str
    save_audio_path: str
    save_image_path: str
    audio_duration: str
    save_video_path: str


class GraphState(TypedDict):
    story_theme: str
    generated_story:str
    scene_list: list[ScenesList]
    supporting_characters: list[SupportingCharacters]
    main_characters: list[MainCharacters]
    pre_processing_video_path: str
    combined_audio_path: str
    voices_folder: str
    images_folder: str
    videos_folder: str
    messages: Annotated[Sequence[BaseMessage], add_messages]


class SubState(TypedDict):
    current_scene: ScenesList
    output_folder: str


def generate_story_char(state: GraphState) -> GraphState:
    story = state["generated_story"]
    new_prompt = """Based on the Story

    {story},

    create a brief description of the main and supporting character, object, or scene. Include specific details about appearance, characteristics . This description will be used to maintain consistency across multiple scenes.
    and also generate a narration that exactly follows this text, starting with 'Once upon a time...' and keeping all sentences unchanged. Do not change the wording or style.
    Extract structured JSON data from this story: 
    
    Return only valid JSON (without markdown formatting). Do NOT wrap it in triple backticks (```json). Ensure it follows this structure:

    {{
      "main_characters": [
              {{
                "name": "",
                "appearance": "",
                "characteristics": ""
              }}
            ],
            "supporting_characters": [
              {{
                "name": "",
                "appearance": "",
                "characteristics": ""
              }}
            ],
          "scenes": [
            {{
              "id": "1",
              "scene": "Engaging Beginning",
              "description": "Begin with a captivating moment to grab children's attention.",
              "narration": ""
              "object_description": ""
            }},
         
          ],   
    }}

    Strictly output **only JSON** without extra text.
    """

    sys_msg = SystemMessage(
        content="You are an assistant that extracts all character names and details from a story."
    )
    hum_msg = HumanMessage(content=new_prompt.format(story=story))
    print(f"Before generating the story characters")
    start_time = time.time()
    res = llm.invoke([sys_msg, hum_msg])
    end_time = time.time()
    print(f"Time taken to generate story characters: {end_time - start_time} seconds")
    print(f"After generating the story characters:{res}")
    parsed_resp = json.loads(res.content)
    print(f"Parsed response: {parsed_resp}")
    for scene in parsed_resp["scenes"]:

        prompt_template = f"""Create a detailed, photorealistic image of the following scene 
        {scene["description"]} 

        **Mood & Lighting**: Cinematic, immersive atmosphere, realistic lighting to match the scene's emotions.

        Ensure character consistency throughout all images. The illustration should capture the story’s essence and atmosphere.
        """
        scene["img_prompt"] = prompt_template

    state["scene_list"] = parsed_resp["scenes"] 
    state["supporting_characters"] = parsed_resp["supporting_characters"]
    state["main_characters"] = parsed_resp["main_characters"]
    return state


workflow = StateGraph(GraphState)
workflow.add_node("generate_story_char", generate_story_char)
workflow.add_edge(START, "generate_story_char")
workflow.add_edge("generate_story_char", END)
app = workflow.compile()

resp2 = app.invoke({"generated_story": resp1["generated_story"]})

for scene in resp2["scene_list"]:
    display(Markdown(f"\n{'='*40}"))
    display(Markdown(f"### Scene {scene['id']}: {scene['scene']}"))
    display(Markdown(f"---"))
    display(Markdown(f"**📖 Description:** {scene['description']}"))
    display(Markdown(f"\n**📜 Narration:**\n{scene['narration']}"))
    display(Markdown(f"\n**🖼️ Image Prompt:**\n{scene['img_prompt']}"))
    display(Markdown(f"{'='*40}\n"))

Before generating the story characters
Time taken to generate story characters: 7.874716758728027 seconds
After generating the story characters:content='{\n  "main_characters": [\n    {\n      "name": "Barnaby Bunson",\n      "appearance": "fluffy white tail, white fur",\n      "characteristics": "incredibly fast, confident, smug, learns to appreciate perseverance"\n    },\n    {\n      "name": "Sheldon",\n      "appearance": null,\n      "characteristics": "tortoise, slow, wise, steady, persistent"\n    }\n  ],\n  "supporting_characters": [\n    {\n      "name": "Mr. Fox",\n      "appearance": null,\n      "characteristics": "race official"\n    },\n    {\n      "name": "Mrs. Badger",\n      "appearance": null,\n      "characteristics": null\n    },\n    {\n      "name": "Grumpy Old Owl",\n      "appearance": null,\n      "characteristics": null\n    }\n  ],\n  "scenes": [\n    {\n      "id": "1",\n      "scene": "Sunny Meadow Introduction",\n      "description": "Introduces Barnaby a


========================================

### Scene 1: Sunny Meadow Introduction

---

**📖 Description:** Introduces Barnaby and Sheldon in Sunny Meadow.

...

---

**📖 Description:** Barnaby learns a valuable lesson about perseverance.


**📜 Narration:**
Barnaby was surprised and a little embarrassed.  He learned a valuable lesson that day: steady persistence can beat even the fastest speed.  From that day on, Barnaby was still fast, but he also learned to appreciate the importance of perseverance and not to underestimate anyone, no matter how slow they seemed.  And Sheldon?  Well, Sheldon became a legend in Sunny Meadow, a reminder that slow and steady wins the race.


**🖼️ Image Prompt:**
Create a detailed, photorealistic image of the following scene 
        Barnaby learns a valuable lesson about perseverance. 

        **Mood & Lighting**: Cinematic, immersive atmosphere, realistic lighting to match the scene's emotions.

        Ensure character consistency throughout all images. The illustration should capture the story’s essence and atmosphere.
        

========================================


In [5]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from datetime import datetime
import os
import nest_asyncio
import time
from gtts import gTTS
from typing import Sequence
from langchain_core.messages import BaseMessage, HumanMessage
from typing import Annotated
from langgraph.graph.message import add_messages
from typing import Any
from langgraph.types import Send
import asyncio
import edge_tts
import pyttsx3
from moviepy import *
nest_asyncio.apply()


class MainCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class SupportingCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class ScenesList(TypedDict):
    id: str
    scene: str
    description: str
    narration: str
    img_prompt: str
    object_description: str
    save_audio_path: str
    save_image_path: str
    audio_duration: str
    save_video_path: str


class GraphState(TypedDict):
    story_theme: str
    generated_story:str
    scene_list: list[ScenesList]
    supporting_characters: list[SupportingCharacters]
    main_characters: list[MainCharacters]
    pre_processing_video_path: str
    combined_audio_path: str
    voices_folder: str
    images_folder: str
    videos_folder: str
    messages: Annotated[Sequence[BaseMessage], add_messages]


class SubState(TypedDict):
    current_scene: ScenesList
    output_folder: str


async def save_voice(save_path, current_voice):
    await current_voice.save(save_path)


async def call_llm_gen_voice(voice):
    try:
        tts = edge_tts.Communicate(
            voice, voice="en-US-JennyNeural", volume="+100%", pitch="+5Hz"
        )
        return tts
    except Exception as e:
        raise ValueError("Error while generating the voice", e)


async def generate_scene_voice(state: SubState):
    cur_scene = state["current_scene"]
    scene_id = int(cur_scene["id"])
    output_folder = state["output_folder"]
    narration_text = cur_scene.get("narration", "")

    try:
        print(f"Generating voice for scene {scene_id}")
        generated_voice = await call_llm_gen_voice(narration_text)
        save_path = os.path.join(output_folder, f"voice_scene_{scene_id}.mp3")
        save_time = time.time()

        await save_voice(save_path, generated_voice)
        end_save_time = time.time()
        
        print(f"saving time for this scene {scene_id}={end_save_time-save_time}")
        print(f"Generated voice successfully for scene {scene_id}")
        try:
            audio_duration = AudioFileClip(save_path).duration
        except Exception as e:
            print(f"Error generating voice for scene {scene_id}: {e}")  
        message = HumanMessage(
            content="",
            additional_kwargs={
                "voice_id": scene_id,
                "save_path": save_path,
                "audio_duration": audio_duration
            }
        )
        
        return {"messages": [message]}

    except Exception as e:
        print(f"Error generating voice for scene {scene_id}: {e}")


def continue_generate_voice(state: GraphState):
    scene_list = state["scene_list"]
    time_stamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder = f"voices_{time_stamp}"
    output_folder = os.path.join("New_Generated_voices", dynamic_folder)
    output_folder = os.path.join("arman_output", output_folder)
    os.makedirs(output_folder, exist_ok=True)
    return [
        Send(
            "generate_scene_voice",
            {
                "current_scene": scene,
                "output_folder": output_folder,
            },
        )
        for scene in scene_list
    ]


def final_node(state: GraphState):
    total_messages = state["messages"]
    print(f"Toatal messages:{total_messages}")
    scene_list = state["scene_list"]
    list_voice_folder = set()
    for scene in scene_list:
        get_voiceid = int(scene["id"])
        for msg in total_messages:

            if get_voiceid == int(msg.additional_kwargs.get("voice_id")):
                print(f"{get_voiceid} {msg}")
                scene["save_audio_path"] = msg.additional_kwargs.get("save_path", "")
                scene["audio_duration"] = msg.additional_kwargs.get("audio_duration", "")
                list_voice_folder.add(os.path.dirname(msg.additional_kwargs.get("save_path", "")))
                
                break
    state["voices_folder"] = list(list_voice_folder)[0] if list_voice_folder else "" 
    print(f'voices folder :{list(list_voice_folder)[0] if list_voice_folder else "" }')
    return state
      


def sync_generate_scene_voice(state: SubState):
    return asyncio.run(generate_scene_voice(state))


workflow = StateGraph(GraphState)
# workflow.add_node("generate_scene_voice", generate_scene_voice)
workflow.add_node("generate_scene_voice", sync_generate_scene_voice)
workflow.add_node("final_node", final_node)

workflow.add_conditional_edges(START, continue_generate_voice, ["generate_scene_voice"])
workflow.add_edge("generate_scene_voice", "final_node")
workflow.add_edge("final_node", END)

app = workflow.compile()


async def run_workflow():
    response = await app.ainvoke({"scene_list": resp2["scene_list"]})
    
    print(response)
    return response

loop = asyncio.get_event_loop()

respe_data=loop.run_until_complete(run_workflow())

Generating voice for scene 2
Generating voice for scene 1
Generating voice for scene 3
Generating voice for scene 5
Generating voice for scene 6
Generating voice for scene 4
saving time for this scene 2=2.7328569889068604
Generated voice successfully for scene 2
saving time for this scene 1=2.9246037006378174
Generated voice successfully for scene 1
saving time for this scene 4=3.01371431350708
Generated voice successfully for scene 4
saving time for this scene 6=3.02935791015625
Generated voice successfully for scene 6
saving time for this scene 5=3.6500744819641113
Generated voice successfully for scene 5
saving time for this scene 3=5.309618949890137
Generated voice successfully for scene 3
Toatal messages:[HumanMessage(content='', additional_kwargs={'voice_id': 1, 'save_path': 'arman_output\\New_Generated_voices\\voices_20250405105330\\voice_scene_1.mp3', 'audio_duration': 20.33}, response_metadata={}, id='4cd957a0-0df6-448a-a4c3-446b3ea5c962'), HumanMessage(content='', additional_

In [6]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from datetime import datetime
import os
from gtts import gTTS 
from typing import Sequence
from langchain_core.messages import BaseMessage, HumanMessage
from typing import Annotated
from langgraph.graph.message import add_messages
from typing import Any
import logging
from langgraph.types import Send
import requests
import random


class MainCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class SupportingCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class ScenesList(TypedDict):
    id: str
    scene: str
    description: str
    narration: str
    img_prompt: str
    object_description: str
    save_audio_path: str
    save_image_path: str
    audio_duration: str
    save_video_path: str


class GraphState(TypedDict):
    story_theme: str
    generated_story:str
    scene_list: list[ScenesList]
    supporting_characters: list[SupportingCharacters]
    main_characters: list[MainCharacters]
    pre_processing_video_path: str
    combined_audio_path: str
    voices_folder: str
    images_folder: str
    videos_folder: str
    messages: Annotated[Sequence[BaseMessage], add_messages]


class SubState(TypedDict):
    current_scene: ScenesList
    output_folder: str



def saveImage(image_content, save_path):
    output_folder = os.path.dirname(save_path)
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    with open(save_path, "wb") as f:
        f.write(image_content)


def images_generates(prompt: str):
    try: 
        color_value1 = format(random.randint(0, 16777215), '06x')  
        color_value2 = format(random.randint(0, 16777215), '06x')
        image_url = f"https://placehold.co/1080x1920/{color_value1}/{color_value2}/png" 
        return image_url 

    except Exception as e:
        logging.error(f"Unexpected error generating image: {e}")
        raise


 

def generate_scence_image(state:SubState):
    cur_scene = state["current_scene"]
    scene_id = int(cur_scene["id"])
    output_folder = state["output_folder"]  
    image_prompt = cur_scene.get("Img_prompt", "")
    try:
        print(f"Generating image for scene {scene_id}")
        imageUrl = images_generates(image_prompt)
        if not imageUrl:
            raise ValueError("Generated image URL is empty.")
        imageContent = requests.get(imageUrl).content
        if not imageContent:
            raise ValueError("Failed to retrieve image content.")
        save_path = os.path.join(output_folder, f"image_scene_{scene_id}.png")
        saveImage(imageContent, save_path)
        print(f"Images saved to {save_path}")
        message = HumanMessage(content='',additional_kwargs={"image_id": scene_id,"save_path": save_path})
        return {'messages':[message]}

    except Exception as e:
        logging.error(f"Error processing scene {scene_id}: {e}")
       
def final_node(state:GraphState):
    total_messages = state['messages'] 
    scene_list = state['scene_list'] 
    image_list =set()
    for scene in scene_list:
        get_imageid = scene['id'] 
        for msg in total_messages:
            if int(get_imageid)== msg.additional_kwargs.get('image_id'): 
                scene['save_image_path'] = msg.additional_kwargs.get('save_path','')
                image_list.add(os.path.dirname(msg.additional_kwargs.get('save_path','')))
                break        
    state['images_folder']= list(image_list)[0] if image_list else ""  
    
    return state      
    # return {'Scene_list':scene_list}
     

def continue_scence_image(state:GraphState):
    scene_list = state['scene_list']
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder = f"images_{timestamp}"
    output_folder = os.path.join("New_Generated_images", dynamic_folder)
    output_folder = os.path.join("arman_output", output_folder)
    os.makedirs(output_folder, exist_ok=True) 
    return [Send("generate_scence_image",{'current_scene':value, 'output_folder':output_folder}) for value in scene_list ]
     
     
     
     
     
     
workflow = StateGraph(GraphState)
workflow.add_node('generate_scence_image',generate_scence_image)
workflow.add_node('final_node',final_node)
 

workflow.add_conditional_edges(START,continue_scence_image,['generate_scence_image'])
workflow.add_edge('generate_scence_image','final_node')
workflow.add_edge('final_node',END)
app = workflow.compile()

resp3 = app.invoke({"scene_list": resp2['scene_list']})
print(resp3)

Generating image for scene 1Generating image for scene 2

Generating image for scene 3
Generating image for scene 4
Generating image for scene 5
Generating image for scene 6
Images saved to arman_output\New_Generated_images\images_20250405105346\image_scene_5.png
Images saved to arman_output\New_Generated_images\images_20250405105346\image_scene_2.png
Images saved to arman_output\New_Generated_images\images_20250405105346\image_scene_6.png
Images saved to arman_output\New_Generated_images\images_20250405105346\image_scene_4.png
Images saved to arman_output\New_Generated_images\images_20250405105346\image_scene_3.png
Images saved to arman_output\New_Generated_images\images_20250405105346\image_scene_1.png
{'scene_list': [{'id': '1', 'scene': 'Sunny Meadow Introduction', 'description': 'Introduces Barnaby and Sheldon in Sunny Meadow.', 'narration': 'Barnaby Bunson was a rabbit known for two things: his fluffy white tail and his incredible speed.  He zoomed through Sunny Meadow, a blur of

In [12]:

import cv2
import numpy as np
import os
from datetime import datetime
from moviepy import *
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
from typing import Sequence
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage
from typing import Annotated
from IPython.display import display, Markdown
from PIL import Image, ImageDraw, ImageFont
import math
class ClassObject(TypedDict):
    object: str
    description: str

class MainCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str

class SupportingCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str

class ScenesList(TypedDict):
    id: str
    scene: str
    description: str
    narration: str
    img_prompt: str
    save_audio_path: str
    save_image_path: str
    save_video_path:str
    save_audio_video_path: str
    audio_duration: str
    objects: list[ClassObject]

class GraphState(TypedDict): 
    main_characters: list[MainCharacters]
    supporting_characters: list[SupportingCharacters]
    scene_list: list[ScenesList]
    messages: Annotated[Sequence[BaseMessage], add_messages]
    combine_audio_video_key: Annotated[Sequence[BaseMessage], add_messages]
    videos_with_audio_folder: str
    pre_processing_video_path: str
    voices_folder: str
    images_folder: str
    videos_folder:str
    videos_with_audio_folder:str
    

class SubState(TypedDict): 
    current_scene: ScenesList
    output_folder: str
 
 
def zoom_in(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-in effect."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + (i / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames

def zoom_out(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-out effect without blinking."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + ((num_frames - i - 1) / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames

def fade_in(image, num_frames=2):
    """Creates a fade-in effect from black to the image."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames

def fade_out(image, num_frames=30):
    """Creates a fade-out effect from image to black."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = 1 - i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames

# ----------------------- Pre-processing Video Function -----------------------
def pre_processing_video(state: SubState):
    cur_scene = state['current_scene']
    scene_id = int(cur_scene['id'])
    output_folder = state['output_folder']
    fps = 10    

    first_img = cv2.imread(cur_scene["save_image_path"])
    if first_img is None:
        raise ValueError(f"Cannot load image: {cur_scene['save_image_path']}")
    
    h, w, _ = first_img.shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    pre_processing_video_path = os.path.join(output_folder, f"video_saving_{scene_id}.mp4")
    video_writer = cv2.VideoWriter(pre_processing_video_path, fourcc, fps, (w, h)) 
    
    img_path = cur_scene["save_image_path"]
    duration = float(cur_scene["audio_duration"]) 
    total_frames = math.ceil(duration * fps)

    fade_duration = int(fps * 1)
    core_frames = total_frames - 2 * fade_duration if total_frames > 2 * fade_duration else max(1, total_frames - fade_duration)

    img = cv2.imread(img_path) 
    img = cv2.resize(img, (w, h))

    if scene_id == 0:
        zoom_in_frames = zoom_in(img, num_frames=core_frames, zoom_factor=0.5)
        fade_out_frames = fade_out(zoom_in_frames[-1], num_frames=fade_duration)
        effect_frames = zoom_in_frames + fade_out_frames
    elif scene_id % 2 == 1:
        zoom_out_frames = zoom_out(img, num_frames=core_frames, zoom_factor=0.5)
        fade_out_frames = fade_out(zoom_out_frames[-1], num_frames=fade_duration)
        effect_frames = zoom_out_frames + fade_out_frames
    else:
        fade_in_frames = fade_in(img, num_frames=fade_duration)
        zoom_in_frames = zoom_in(img, num_frames=core_frames, zoom_factor=0.5)
        fade_out_frames = fade_out(zoom_in_frames[-1], num_frames=fade_duration)
        effect_frames = fade_in_frames + zoom_in_frames + fade_out_frames

    # Ensure frame count matches exactly
    if len(effect_frames) > total_frames:
        effect_frames = effect_frames[:total_frames]
    elif len(effect_frames) < total_frames:
        last_frame = effect_frames[-1]
        effect_frames += [last_frame] * (total_frames - len(effect_frames))

    for frame in effect_frames:
        video_writer.write(frame)
    
    video_writer.release()
    print(f"✅ Audio Duration for scene {scene_id}: {duration:.2f} sec")
    
    final_video = VideoFileClip(pre_processing_video_path)
    final_video_duration = final_video.duration
    print(f"✅ Final Video Duration for scene {scene_id}: {final_video_duration:.2f} sec (No Audio)")
    print(f"Pre-processing video saved as {pre_processing_video_path}")
    
    message = HumanMessage(content='', additional_kwargs={"video_id": scene_id, "save_path": pre_processing_video_path})
    return {'messages': [message]}

def save_paths(state: GraphState):
    total_messages = state["messages"]
    print(f"Toatal messages:{total_messages}")
    scene_list = state["scene_list"]
    video_list =set()
    for scene in scene_list:
        get_id = int(scene["id"])
     
        for msg in total_messages:
            video_id = msg.additional_kwargs.get('video_id') 
            
            if video_id is not None and get_id == video_id: 
                save_path = msg.additional_kwargs.get('save_path','')
                print(f"{get_id} {msg}")
                scene["save_video_path"] = save_path 
                video_list.add(os.path.dirname(save_path)) 
    state['videos_folder'] =list(video_list)[0]  if video_list else ""          
    return state

def continue_generate_images(state: GraphState):
    scene_list = state['scene_list']
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder = f"videolist_{timestamp}"
    output_folder = os.path.join("video_list", dynamic_folder)
    output_folder = os.path.join("arman_output", output_folder)
    os.makedirs(output_folder, exist_ok=True) 
    return [Send('pre_processing_video', {'current_scene':value,'output_folder':output_folder}) for value in scene_list]

def combine_audio_with_video(state: SubState):
    cur_scene = state['current_scene']
    scene_id = int(cur_scene['id'])
    video_path = cur_scene['save_video_path']
    audio_path = cur_scene['save_audio_path']
    output_folder = state['output_folder']
 
    
    combining_audio_with_video_path = os.path.join(output_folder, f"video_with_audio_{scene_id}.mp4")
    !ffmpeg -i "{video_path}" -i "{audio_path}" -c:v copy -c:a aac -strict experimental "{combining_audio_with_video_path}"
    print(f'audio_path==================={audio_path}')
    print(f'output_folder==================={output_folder}')
    message = HumanMessage(content='', additional_kwargs={"scene_id": scene_id, "save_path": combining_audio_with_video_path})
    return {"combine_audio_video_key":[message]}
def final_node(state: GraphState):
    total_messages = state["combine_audio_video_key"]
    print(f"Toatal messages:{total_messages}")
    scene_list = state["scene_list"]
    video_list =set()
    for scene in scene_list:
        get_id = int(scene["id"])
     
        for msg in total_messages:
            scene_id = msg.additional_kwargs.get('scene_id') 
            
            if scene_id is not None and get_id == scene_id: 
                save_path = msg.additional_kwargs.get('save_path','')
                print(f"{get_id} {msg}")
                scene["save_audio_video_path"] = save_path 
                video_list.add(os.path.dirname(save_path)) 
    state['videos_with_audio_folder'] =list(video_list)[0]  if video_list else ""          
    return state
def combining_videos(state:GraphState):
    scene_list = state['scene_list'] 
    sorted_scene = sorted(scene_list, key=lambda x: int(x['id']))
    video_files = [scene['save_audio_video_path'] for scene in sorted_scene]
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S") 
    output_folder = os.path.join("arman_output", 'combined_videos') 
    os.makedirs(output_folder, exist_ok=True)
    concat_txt_path = "arman_output/concat_list.txt"
    with open(concat_txt_path, "w") as f:
        for file in video_files:
            f.write(f"file '{os.path.abspath(file)}'\n")
    import subprocess
    final_output_video = os.path.join(output_folder, f"combined_video_{timestamp}.mp4")
    command = [
    "ffmpeg",
    "-f", "concat",
    "-safe", "0",
    "-i", concat_txt_path,
    "-c", "copy",
    final_output_video
]

    

    process = subprocess.run(command, capture_output=True, text=True)
    state['pre_processing_video_path'] = final_output_video
    print("FFmpeg stdout:", process.stdout)
    print("FFmpeg stderr:", process.stderr)
from pydub import AudioSegment
  
def merge_single_audiovideo(state:GraphState):
    scene_list = state['scene_list'] 
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder = os.path.join('videos_with_audio', f"video_with_audio_{timestamp}")
    output_folder = os.path.join("arman_output", dynamic_folder)
    os.makedirs(output_folder, exist_ok=True)
    return [Send('combine_audio_with_video', {'current_scene':value,'output_folder':output_folder}) for value in scene_list]

workflow = StateGraph(GraphState)
workflow.add_node('pre_processing_video', pre_processing_video)
workflow.add_node('combine_audio_with_video', combine_audio_with_video)
workflow.add_node('save_paths', save_paths)
workflow.add_node('final_node', final_node)
workflow.add_node('combining_videos', combining_videos)
# workflow.add_node('finaliaz_video', finaliaz_video)


workflow.add_conditional_edges(START,continue_generate_images,['pre_processing_video'])
workflow.add_edge('pre_processing_video','save_paths')
workflow.add_conditional_edges('save_paths',merge_single_audiovideo,['combine_audio_with_video'])
# workflow.add_edge('combine_audio_with_video','final_node')
# workflow.add_edge('final_node','combining_videos') 
# workflow.add_edge('combining_videos',END)
 
app = workflow.compile() 
resp4 = app.invoke({"scene_list": resp3['scene_list']})
print(resp4)


✅ Audio Duration for scene 2: 9.96 sec
{'video_found': True, 'audio_found': False, 'metadata': {'major_brand': 'isom', 'minor_version': '512', 'compatible_brands': 'isomiso2mp41', 'encoder': 'Lavf58.76.100'}, 'inputs': [{'streams': [{'input_number': 0, 'stream_number': 0, 'stream_type': 'video', 'language': None, 'default': True, 'size': [1080, 1920], 'bitrate': 721, 'fps': 10.0, 'codec_name': 'mpeg4', 'profile': '(Simple Profile)', 'metadata': {'handler_name': 'VideoHandler'}}], 'input_number': 0}], 'duration': 10.0, 'bitrate': 722, 'start': 0.0, 'default_video_input_number': 0, 'default_video_stream_number': 0, 'video_codec_name': 'mpeg4', 'video_profile': '(Simple Profile)', 'video_size': [1080, 1920], 'video_bitrate': 721, 'video_fps': 10.0, 'video_duration': 10.0, 'video_n_frames': 100}
c:\Users\arman\AppData\Local\Programs\Python\Python312\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win64-v4.2.2.exe -i arman_output\video_list\videolist_20250405112124\video_saving_2.mp4 -logl

ffmpeg version 2025-03-20-git-76f09ab647-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 14.2.0 (Rev1, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-libgme --enable-libopenmpt --enable-libopencore-amrwb -

audio_path===================arman_output\New_Generated_voices\voices_20250405105330\voice_scene_4.mp3
output_folder===================arman_output\videos_with_audio\video_with_audio_20250405112138
audio_path===================arman_output\New_Generated_voices\voices_20250405105330\voice_scene_5.mp3
output_folder===================arman_output\videos_with_audio\video_with_audio_20250405112138
audio_path===================arman_output\New_Generated_voices\voices_20250405105330\voice_scene_6.mp3
output_folder===================arman_output\videos_with_audio\video_with_audio_20250405112138
audio_path===================arman_output\New_Generated_voices\voices_20250405105330\voice_scene_1.mp3
output_folder===================arman_output\videos_with_audio\video_with_audio_20250405112138


ffmpeg version 2025-03-20-git-76f09ab647-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 14.2.0 (Rev1, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-libgme --enable-libopenmpt --enable-libopencore-amrwb -

audio_path===================arman_output\New_Generated_voices\voices_20250405105330\voice_scene_3.mp3
output_folder===================arman_output\videos_with_audio\video_with_audio_20250405112138
{'scene_list': [{'id': '1', 'scene': 'Sunny Meadow Introduction', 'description': 'Introduces Barnaby and Sheldon in Sunny Meadow.', 'narration': 'Barnaby Bunson was a rabbit known for two things: his fluffy white tail and his incredible speed.  He zoomed through Sunny Meadow, a blur of white fur, leaving other animals in his dust. One sunny afternoon, Barnaby hopped past Sheldon the tortoise, who was slowly, slowly making his way towards a juicy dandelion.', 'object_description': 'Sunny Meadow, juicy dandelion', 'img_prompt': "Create a detailed, photorealistic image of the following scene \n        Introduces Barnaby and Sheldon in Sunny Meadow. \n\n        **Mood & Lighting**: Cinematic, immersive atmosphere, realistic lighting to match the scene's emotions.\n\n        Ensure character cons

ffmpeg version 2025-03-20-git-76f09ab647-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 14.2.0 (Rev1, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-libgme --enable-libopenmpt --enable-libopencore-amrwb -

In [17]:
from pydub import AudioSegment

# Load narration and background music
narration = AudioSegment.from_file('arman_output\\New_Generated_voices\\voices_20250405105330\\voice_scene_4.mp3')
bg_music = AudioSegment.from_file("bg_music.mp3")

# Reduce the volume of background music (further decrease if needed)
bg_music = bg_music - 35  # Try a stronger reduction if needed

# Loop the background music to match the narration length
while len(bg_music) < len(narration):
    bg_music += bg_music    

bg_music = bg_music[:len(narration)]  # Trim the background music to match narration length

# Optionally, you can reduce the volume of the narration if needed
   # Increase narration volume if needed

# Overlay the background music on the narration
merged_audio = narration.overlay(bg_music)

# Export the final merged audio
merged_audio.export('armanaliali.mp3', format="mp3")

print("✅ Merged audio with background music saved.")


✅ Merged audio with background music saved.


In [13]:
# Combining videos in one video
import os

video_files = [
    "arman_output/videos_with_audio/video_with_audio_20250405084906/video_with_audio_1.mp4",
    "arman_output/videos_with_audio/video_with_audio_20250405084906/video_with_audio_2.mp4",
    "arman_output/videos_with_audio/video_with_audio_20250405084906/video_with_audio_3.mp4",
    "arman_output/videos_with_audio/video_with_audio_20250405084906/video_with_audio_4.mp4",
]

concat_txt_path = "arman_output/concat_list.txt"
with open(concat_txt_path, "w") as f:
    for file in video_files:
        f.write(f"file '{os.path.abspath(file)}'\n")
import subprocess

output_video = "arman_output/final_story_video_with_audio.mp4"

command = [
    "ffmpeg",
    "-f", "concat",
    "-safe", "0",
    "-i", concat_txt_path,
    "-c", "copy",
    output_video
]

process = subprocess.run(command, capture_output=True, text=True)
print("FFmpeg stdout:", process.stdout)
print("FFmpeg stderr:", process.stderr)


FFmpeg stdout: 
FFmpeg stderr: ffmpeg version 2025-03-20-git-76f09ab647-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 14.2.0 (Rev1, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-libgme --enable-libopenm

In [15]:
import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont

def split_text_into_segments(text, font, max_width):
    words = text.split()
    segments = []
    current_segment = ""
    for word in words:
        test_line = current_segment + (" " if current_segment else "") + word
        w = font.getbbox(test_line)[2]
        if w <= max_width:
            current_segment = test_line
        else:
            if current_segment:
                segments.append(current_segment)
            current_segment = word
    if current_segment:
        segments.append(current_segment)
    print(f"print segments: {segments}")    
    return segments

def zoom_in(image, num_frames=30, zoom_factor=0.5):
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + (i / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames

def zoom_out(image, num_frames=30, zoom_factor=0.5):
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + ((num_frames - i - 1) / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames

def create_video_with_typewriter_effect(scene_data, output_filename, fontsize=60, video_fps=15):
    font = ImageFont.truetype("arial.ttf", fontsize)
    video_width = 1080
    video_height = 1920
    video_size = (video_width, video_height)
    video = cv2.VideoWriter(output_filename, 
                            cv2.VideoWriter_fourcc(*'mp4v'), 
                            video_fps, 
                            video_size)
    
    for idx, scene in enumerate(scene_data):
        narration = scene['Narration']
        scene_duration = scene['Audio_duration']
        image_path = scene.get('Save_image_path', None)
        num_frames = int(scene_duration * video_fps)
        
        if image_path:
            image = cv2.imread(image_path)
            image = cv2.resize(image, (video_width, video_height))
            frames = zoom_in(image, num_frames) if idx % 2 == 0 else zoom_out(image, num_frames)
        else:
            frames = [np.zeros((video_height, video_width, 3), dtype=np.uint8) for _ in range(num_frames)]
        
        segments = split_text_into_segments(narration, font, video_width - 40)
        num_segments = len(segments)
        if num_segments == 0:
            continue
        
        total_chars = sum(len(segment) for segment in segments)
        
        # Allocate time to each segment proportionally to its length
        for segment in segments:
            seg_chars = len(segment)
            segment_duration = (seg_chars / total_chars) * scene_duration
            segment_frames = int(segment_duration * video_fps)
            total_chars_in_segment = len(segment)
            for frame_idx in range(segment_frames):
                frame = frames[frame_idx % num_frames].copy()
                frame_pil = Image.fromarray(frame)
                draw = ImageDraw.Draw(frame_pil)
                
                # Ensure final frame shows full text
                char_count = int(((frame_idx + 1) / segment_frames) * total_chars_in_segment)
                displayed_text = segment[:char_count]
                
                text_width, text_height = font.getbbox(displayed_text)[2:4]
                x = (video_width - text_width) // 2
                y = (video_height - text_height) // 2
                
                # Draw a black rectangle as background behind the text
                padding = 10
                rect_coords = [(x - padding, y - padding), (x + text_width + padding, y + text_height + padding)]
                draw.rectangle(rect_coords, fill=(0, 0, 0))
                
                # Now draw the text on top
                draw.text((x, y), displayed_text, font=font, fill=(255, 255, 255))
                
                video.write(np.array(frame_pil))
    
    video.release()
    print(f"Video saved as {output_filename}")

scene_data = [
    {'id': '1',
     'Scene': 'Sunny Meadow',
     'Description': 'Sunny Meadow where Barnaby and Sheldon meet near the big oak tree.',
     'Narration': 'One sunny morning, Barnaby hopped past Sheldon Shelldon, a tortoise whose shell was a beautiful shade of deep green. Sheldon was slowly, slowly making his way to the big oak tree   ',
     'Save_audio_path': 'arman_output\\New_Generated_voices\\voices_20250403064254\\voice_scene_1.mp3',
     'Audio_duration': 12.55,
     'Save_image_path': 'arman_output\\New_Generated_images\\images_20250403064306\\image_scene_1.png'},
]

output_filename = "typewriter_animatiosssssssssn.mp4"
create_video_with_typewriter_effect(scene_data, output_filename)


print segments: ['One sunny morning, Barnaby hopped', 'past Sheldon Shelldon, a tortoise', 'whose shell was a beautiful shade of', 'deep green. Sheldon was slowly,', 'slowly making his way to the big oak', 'tree']
Video saved as typewriter_animatiosssssssssn.mp4


In [24]:

import cv2
import numpy as np
import os
from datetime import datetime
from moviepy import *
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
from typing import Sequence
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage
from typing import Annotated
from IPython.display import display, Markdown
class ClassObject(TypedDict):
    Object: str
    Description: str
    
class MainCharacters(TypedDict):
    Name: str
    Appearance: str
    Characteristics: str
    
class SupportingCharacters(TypedDict):
    Name: str
    Appearance: str
    Characteristics: str
    
class ScenesList(TypedDict):
    id: str
    Scene: str
    Description: str
    Narration: str
    Img_prompt: str
    Save_audio_path:str
    Save_image_path:str    
    
class GraphState(TypedDict): 
    MainCharacters:list[MainCharacters]
    SupportingCharacters: list[SupportingCharacters]
    Scene_list: list[ScenesList]
    Objects: list[ClassObject]
    messages: Annotated[Sequence[BaseMessage], add_messages]
    pre_processing_video_path:str
    Voices_folder:str
    Images_folder:str
    
class SubState(TypedDict): 
    current_scene: ScenesList
    output_folder: str

def zoom_in(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-in effect."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + (i / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames


def zoom_out(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-out effect without blinking."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + ((num_frames - i - 1) / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames


def fade_in(image, num_frames=2):
    """Creates a fade-in effect from black to the image."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames


def fade_out(image, num_frames=30):
    """Creates a fade-out effect from image to black."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = 1 - i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames
def pre_processing_video(state:GraphState):
    fps=2
    
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder=f"pre_video_{timestamp}"
    output_folder=os.path.join("Pre_Generated_videos",dynamic_folder)
    output_folder=os.path.join("output",output_folder)
    os.makedirs(output_folder, exist_ok=True)
    file_name = f'pre_video_{timestamp}.mp4'
    
    pre_processing_video_path=os.path.join(output_folder,file_name)
    scene_list=state['Scene_list']
    if not scene_list:
        raise ValueError("No scenes provided.")

    first_img = cv2.imread(scene_list[0]["Save_image_path"])
    if first_img is None:
        raise ValueError(f"Cannot load image: {scene_list[0]['Save_image_path']}")

    h, w, _ = first_img.shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video_writer = cv2.VideoWriter(pre_processing_video_path, fourcc, fps, (w, h))

    for idx, scene in enumerate(scene_list):
        img_path = scene["Save_image_path"]
        duration = float(scene["Audio_duration"])

        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Cannot load image {img_path}. Skipping.")
            continue
        img = cv2.resize(img, (w, h))

        total_frames = int(fps * duration)
        fade_in_frame = int(fps * 1)
        fade_out_frame = int(fps * 1)
        first_image_duration = total_frames - fade_out_frame

        if idx == 0:
            zoom_in_frames = zoom_in(img, num_frames=first_image_duration, zoom_factor=0.5)
            final_zoom_frame = zoom_in_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = zoom_in_frames + fade_out_frames
        elif idx % 2 == 1:
            zoom_out_frames = zoom_out(img, num_frames=first_image_duration, zoom_factor=0.5)
            final_zoom_frame = zoom_out_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = zoom_out_frames + fade_out_frames
        else:
            fade_in_frames = fade_in(img, num_frames=fade_in_frame)
            zoom_in_frames = zoom_in(img, num_frames=first_image_duration, zoom_factor=0.5)
            final_zoom_frame = zoom_in_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = fade_in_frames + zoom_in_frames + fade_out_frames

        for frame in effect_frames:
            # text = "PositivePlus"
            # font = cv2.FONT_HERSHEY_SIMPLEX
            # font_scale = 1
            # font_thickness = 2
            # color = (220, 255, 255)
            # text_size = cv2.getTextSize(text, font, font_scale, font_thickness)[0]
            # text_x = w - text_size[0] - 20
            # text_y = 60
            # cv2.putText(frame, text, (text_x, text_y), font, font_scale, color, font_thickness, cv2.LINE_AA)

            video_writer.write(frame)

    video_writer.release()
    
    print(f"pre_processing_video saved as {pre_processing_video_path}") 
    state['pre_processing_video_path']  =pre_processing_video_path 
    
    return state
    # return {'pre_processing_video_path':pre_processing_video_path}


     
workflow = StateGraph(GraphState)
workflow.add_node('pre_processing_video',pre_processing_video)
 
  
workflow.add_edge(START,'pre_processing_video')
workflow.add_edge('pre_processing_video',END)
app = workflow.compile()

resp4 = app.invoke({"Scene_list": resp3['Scene_list']})
print(resp4)

pre_processing_video saved as output\Pre_Generated_videos\pre_video_20250403081204\pre_video_20250403081204.mp4
{'Scene_list': [{'id': '1', 'Scene': "Farmer McGregor's field", 'Description': 'Dusty track, old oak tree at the edge of the field, patch of juicy clover.', 'Narration': 'Barnaby Bunson was a rabbit known for two things: his fluffy white tail and his incredible speed.  He zoomed across Farmer McGregor\'s field like a furry blur, leaving dust clouds in his wake.  One sunny morning, Barnaby hopped past a tortoise named Sheldon slowly making his way along the dusty track. "Well, well, well," Barnaby chuckled, his nose twitching. "Look at you, Sheldon!  You\'re slower than a snail in molasses!" Sheldon, who was carefully carrying a tiny, brightly colored flower, stopped and looked up at Barnaby. His wise old eyes twinkled.  "Perhaps," he said calmly, "but speed isn\'t everything.  How about a race to the old oak tree at the edge of the field? The winner gets to keep this flower."

In [31]:

import cv2
import numpy as np
import os
from datetime import datetime
from moviepy import *
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
from typing import Sequence
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage
from typing import Annotated
from IPython.display import display, Markdown
from PIL import Image, ImageDraw, ImageFont
 
class ClassObject(TypedDict):
    Object: str
    Description: str

class MainCharacters(TypedDict):
    Name: str
    Appearance: str
    Characteristics: str

class SupportingCharacters(TypedDict):
    Name: str
    Appearance: str
    Characteristics: str

class ScenesList(TypedDict):
    id: str
    Scene: str
    Description: str
    Narration: str
    Img_prompt: str
    Save_audio_path: str
    Save_image_path: str

class GraphState(TypedDict): 
    MainCharacters: list[MainCharacters]
    SupportingCharacters: list[SupportingCharacters]
    Scene_list: list[ScenesList]
    Objects: list[ClassObject]
    messages: Annotated[Sequence[BaseMessage], add_messages]
    pre_processing_video_path: str
    Voices_folder: str
    Images_folder: str

class SubState(TypedDict): 
    current_scene: ScenesList
    output_folder: str
 
def split_text_into_segments(text, font, max_width):
    """Split the text into segments that fit within max_width."""
    words = text.split()
    segments = []
    current_segment = ""
    for word in words:
        test_line = current_segment + (" " if current_segment else "") + word
        w = font.getbbox(test_line)[2]
        if w <= max_width:
            current_segment = test_line
        else:
            if current_segment:
                segments.append(current_segment)
            current_segment = word
    if current_segment:
        segments.append(current_segment)
    print(f"print segments: {segments}")    
    return segments

def zoom_in(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-in effect."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + (i / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames

def zoom_out(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-out effect without blinking."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + ((num_frames - i - 1) / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames

def fade_in(image, num_frames=2):
    """Creates a fade-in effect from black to the image."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames

def fade_out(image, num_frames=30):
    """Creates a fade-out effect from image to black."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = 1 - i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames

# ----------------------- Pre-processing Video Function -----------------------
def pre_processing_video(state: GraphState):
    fps = 10  # You can adjust FPS as needed
    
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder = f"pre_video_{timestamp}"
    output_folder = os.path.join("Pre_Generated_videos", dynamic_folder)
    output_folder = os.path.join("output", output_folder)
    os.makedirs(output_folder, exist_ok=True)
    file_name = f'pre_video_{timestamp}.mp4'
    
    pre_processing_video_path = os.path.join(output_folder, file_name)
    scene_list = state['Scene_list']
    if not scene_list:
        raise ValueError("No scenes provided.")

    first_img = cv2.imread(scene_list[0]["Save_image_path"])
    if first_img is None:
        raise ValueError(f"Cannot load image: {scene_list[0]['Save_image_path']}")
    
    h, w, _ = first_img.shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video_writer = cv2.VideoWriter(pre_processing_video_path, fourcc, fps, (w, h))
    
    # Load a font for captions (adjust size as needed)
    caption_font = ImageFont.truetype("arial.ttf", 80)
    # caption_font = ImageFont.truetype("Montserrat-Bold.ttf", 64)
    # Maximum width for caption text (with some margin)
    max_caption_width = w - 40
    
    for idx, scene in enumerate(scene_list):
        img_path = scene["Save_image_path"]
        duration = float(scene["Audio_duration"])
        narration = scene["Narration"]
        
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Cannot load image {img_path}. Skipping.")
            continue
        img = cv2.resize(img, (w, h))
        
        total_frames = int(fps * duration)
        fade_in_frame = int(fps * 1)
        fade_out_frame = int(fps * 1)
        first_image_duration = total_frames - fade_out_frame
        
        # Generate zoom/fade effect frames
        if idx == 0:
            zoom_in_frames = zoom_in(img, num_frames=first_image_duration, zoom_factor=0.5)
            final_zoom_frame = zoom_in_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = zoom_in_frames + fade_out_frames
        elif idx % 2 == 1:
            zoom_out_frames = zoom_out(img, num_frames=first_image_duration, zoom_factor=0.5)
            final_zoom_frame = zoom_out_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = zoom_out_frames + fade_out_frames
        else:
            fade_in_frames = fade_in(img, num_frames=fade_in_frame)
            zoom_in_frames = zoom_in(img, num_frames=first_image_duration, zoom_factor=0.5)
            final_zoom_frame = zoom_in_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = fade_in_frames + zoom_in_frames + fade_out_frames
        
        # Split narration into caption segments using your split_text_into_segments logic
        segments = split_text_into_segments(narration, caption_font, max_caption_width)
        num_segments = len(segments)
        if num_segments == 0:
            segments = [""]
        
        # Allocate frames for each segment proportionally to its character length
        total_chars = sum(len(seg) for seg in segments)
        global_frame_index = 0
        for segment in segments:
            seg_chars = len(segment)
            # Calculate how many frames to allocate for this segment based on its relative length
            segment_frames = int((seg_chars / total_chars) * total_frames)
            # Ensure at least one frame is allocated
            segment_frames = max(segment_frames, 1)
            
            for i in range(segment_frames):
                # Get the corresponding effect frame
                if global_frame_index >= len(effect_frames):
                    break
                frame = effect_frames[global_frame_index].copy()
                frame_pil = Image.fromarray(frame)
                draw = ImageDraw.Draw(frame_pil)
                
                # Typewriter effect: gradually reveal the segment
                char_count = int(((i + 1) / segment_frames) * seg_chars)
                displayed_text = segment[:char_count]
                
                # Measure text size and center it (both horizontally and vertically)
                text_width, text_height = caption_font.getbbox(displayed_text)[2:4]
                x = (w - text_width) // 2
                y = (h - text_height) // 2  # vertical center; adjust if needed
                
                # Draw a black rectangle background behind the text for better visibility
                padding = 10
                draw.rectangle([(x - padding, y - padding), (x + text_width + padding, y + text_height + padding)], fill=(0, 0, 0))
                # Draw the caption text in white
                draw.text((x, y), displayed_text, font=caption_font, fill=(255, 255, 255))
                
                video_writer.write(np.array(frame_pil))
                global_frame_index += 1
    
    video_writer.release()
    print(f"pre_processing_video saved as {pre_processing_video_path}") 
    state['pre_processing_video_path'] = pre_processing_video_path 
    return state

# ----------------------- Workflow -----------------------
workflow = StateGraph(GraphState)
workflow.add_node('pre_processing_video', pre_processing_video)
workflow.add_edge(START, 'pre_processing_video')
workflow.add_edge('pre_processing_video', END)
app = workflow.compile()

# Invoke the workflow with your scene data (assumed to be available in resp3['Scene_list'])
resp4 = app.invoke({"Scene_list": resp3['Scene_list']})
print(resp4)


print segments: ['Barnaby Bunson was a', 'rabbit known for two things:', 'his fluffy white tail and his', 'incredible speed. He', 'zoomed across Farmer', "McGregor's field like a furry", 'blur, leaving dust clouds in', 'his wake. One sunny', 'morning, Barnaby hopped', 'past a tortoise named', 'Sheldon slowly making his', 'way along the dusty track.', '"Well, well, well," Barnaby', 'chuckled, his nose twitching.', '"Look at you, Sheldon!', "You're slower than a snail in", 'molasses!" Sheldon, who', 'was carefully carrying a tiny,', 'brightly colored flower,', 'stopped and looked up at', 'Barnaby. His wise old eyes', 'twinkled. "Perhaps," he said', 'calmly, "but speed isn\'t', 'everything. How about a', 'race to the old oak tree at', 'the edge of the field? The', 'winner gets to keep this', 'flower." He held up the', 'flower proudly. Barnaby', 'laughed. "A race? With', "*you*? That's the funniest", "thing I've heard all day! I'll", 'win before you even reach', 'the first dandelion!" And 

In [4]:
# combine audio files 
import os
from moviepy import *
def voice_loading(voices_root_folder):
    try:
        if not isinstance(voices_root_folder, str):
            raise ValueError("Voice folder path should be a string.")
        voices = []
        for voice in sorted(os.listdir(voices_root_folder)):
            if voice.lower().endswith('.mp3'):
                current_voice=os.path.join(voices_root_folder, voice)
                voices.append(AudioFileClip(current_voice))
        if not voices:
            raise ValueError("No valid MP3 files found in the directory.")
        with concatenate_audioclips(voices) as final_clip:
            output_path = "./final_output/combine_audio41.mp3"
            final_clip.write_audiofile(output_path)
        
        return output_path
    except Exception as e:
        raise ValueError(f"Error in voice_loading function: {e}")
                
voice_generates_path = "final_output/voices_20250404111435"
voice_path_loaded = voice_loading(voice_generates_path)
print(f"output_path: {voice_path_loaded}")

chunk:   2%|▏         | 27/1269 [00:17<13:47,  1.50it/s, now=None]

MoviePy - Writing audio in ./final_output/combine_audio41.mp3


chunk:   2%|▏         | 27/1269 [00:18<14:20,  1.44it/s, now=None]

MoviePy - Done.
output_path: ./final_output/combine_audio41.mp3


In [23]:
 
# Combine audio and video files it is not best it takes more time as compared to ffmpeg
from moviepy import *

video_path = "final_outputs/newnewsingle.mp4"  # Change to your uploaded video file
audio_path = "final_outputs/combine_audio41.mp3"  # Change to your uploaded audio file
output_path = "final_outputs/output_video102222.mp4"

videoclip = VideoFileClip(video_path)
audioclip = AudioFileClip(audio_path) 
new_audioclip = CompositeAudioClip([audioclip])
videoclip.audio = new_audioclip
videoclip.write_videofile(f'{output_path}')


{'video_found': True, 'audio_found': False, 'metadata': {'major_brand': 'isom', 'minor_version': '512', 'compatible_brands': 'isomiso2mp41', 'encoder': 'Lavf58.76.100'}, 'inputs': [{'streams': [{'input_number': 0, 'stream_number': 0, 'stream_type': 'video', 'language': None, 'default': True, 'size': [1024, 1792], 'bitrate': 12486, 'fps': 30.0, 'codec_name': 'mpeg4', 'profile': '(Simple Profile)', 'metadata': {'handler_name': 'VideoHandler'}}], 'input_number': 0}], 'duration': 78.03, 'bitrate': 12487, 'start': 0.0, 'default_video_input_number': 0, 'default_video_stream_number': 0, 'video_codec_name': 'mpeg4', 'video_profile': '(Simple Profile)', 'video_size': [1024, 1792], 'video_bitrate': 12486, 'video_fps': 30.0, 'video_duration': 78.03, 'video_n_frames': 2340}
c:\Users\arman\AppData\Local\Programs\Python\Python312\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win64-v4.2.2.exe -i final_outputs/newnewsingle.mp4 -loglevel error -f image2pipe -vf scale=1024:1792 -sws_flags bicubic -pi

MoviePy - Done.
MoviePy - Writing video final_outputs/output_video102222.mp4



MoviePy - Done !
MoviePy - video ready final_outputs/output_video102222.mp4


In [17]:
video_path = "final_output/armanresult57777.mp4"  # Change to your uploaded video file
audio_path = "final_output/audio_20250404111440.mp3"  # Change to your uploaded audio file
output_path = "final_output/final_arman22.mp4"

# Merge video and audio
!ffmpeg -i "{video_path}" -i "{audio_path}" -c:v copy -c:a aac -strict experimental "{output_path}"


ffmpeg version 2025-03-20-git-76f09ab647-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 14.2.0 (Rev1, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-libgme --enable-libopenmpt --enable-libopencore-amrwb -

In [ ]:
# Combine audio and video
from ffmpeg import FFmpeg, Progress 
videoPath = "final_outputs/single.mp4"  # Change to your uploaded video file
audioPath = "final_outputs/single.mp3"  # Change to your uploaded audio file
output_path = "final_outputs/output_video1205.mp4"
ffmpeg = (
        FFmpeg()
        .option("y")
        .input(videoPath)
        .input(audioPath)
        .output(
            output_path,
            codec="copy",
        )
    )
    
@ffmpeg.on("progress")
def on_progress(progress: Progress):
    print(progress)


ffmpeg.execute()

Progress(frame=420, fps=0.0, size=32667648, time=datetime.timedelta(seconds=13, microseconds=750000), bitrate=19004.1, speed=652.0)


b''

In [5]:
import cv2
import numpy as np
import os
from moviepy import *


def zoom_in(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-in effect."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + (i / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames


def zoom_out(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-out effect without blinking."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + ((num_frames - i - 1) / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames


def fade_in(image, num_frames=30):
    """Creates a fade-in effect from black to the image."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames


def fade_out(image, num_frames=30):
    """Creates a fade-out effect from image to black."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = 1 - i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames


def load_voices(voices_root_folder):
    try:
        if not isinstance(voices_root_folder, str):
            raise ValueError("Voice folder path should be a string.")
        print(f"voices_root_folder:{voices_root_folder}")
        voices = sorted(
            [
                os.path.join(voices_root_folder, voice)
                for voice in os.listdir(voices_root_folder)
                if voice.lower().endswith(".mp3")
            ]
        )
        return voices
    except Exception as e:
        raise ValueError(f"Error in load_voices function: {e}")


def load_images(img_root_folder):
    try:
        if not isinstance(img_root_folder, str):
            raise ValueError("image folder path should be a string.")

        images = sorted(
            [
                os.path.join(img_root_folder, img)
                for img in os.listdir(img_root_folder)
                if img.lower().endswith((".png", ".jpg", ".jpeg"))
            ]
        )
        return images
    except Exception as e:
        raise ValueError(f"Error in load_function function: {e}")


def images_to_video_with_effects(
    image_paths, output_video, fps=30, durations=None
): 
    if not image_paths:
        raise ValueError("No images provided.")

    first_img = cv2.imread(image_paths[0])
    if first_img is None:
        raise ValueError(f"Cannot load image: {image_paths[0]}")

    h, w, _ = first_img.shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video_writer = cv2.VideoWriter(output_video, fourcc, fps, (w, h))
 
    if durations is None:
        durations = [2] * len(image_paths)  

    for idx, (path, duration) in enumerate(zip(image_paths, durations)):
        img = cv2.imread(path)
        if img is None:
            print(f"Warning: Cannot load image {path}. Skipping.")
            continue
        img = cv2.resize(img, (w, h))

        total_frames = int(fps * duration)
        print(f"Total frames: {total_frames}")
        fade_in_frame = int(fps * 1)
        fade_out_frame = int(fps * 1)

        first_image_duration = total_frames - fade_out_frame
         

        if idx == 0:
            zoom_in_frames = zoom_in(
                img, num_frames=first_image_duration, zoom_factor=0.5
            )
            final_zoom_frame = zoom_in_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = zoom_in_frames + fade_out_frames
        elif idx % 2 == 1:
            zoom_out_frames = zoom_out(
                img, num_frames=first_image_duration, zoom_factor=0.5
            )
            final_zoom_frame = zoom_out_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = zoom_out_frames + fade_out_frames
        else:
            fade_in_frames = fade_in(img, num_frames=fade_in_frame)
            zoom_in_frames = zoom_in(
                img, num_frames=first_image_duration, zoom_factor=0.5
            )
            final_zoom_frame = zoom_in_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = fade_in_frames + zoom_in_frames + fade_out_frames
        print(f"Total frames written: {len(effect_frames)}")

        for frame in effect_frames:
            text = "youtube"
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 1
            font_thickness = 2
            color = (220, 255, 255)

            text_size = cv2.getTextSize(text, font, font_scale, font_thickness)[0]
            text_x = w - text_size[0] - 20
            text_y = 60

            cv2.putText(
                frame, text, (text_x, text_y), font, font_scale, color, font_thickness
            )

            video_writer.write(frame)

    video_writer.release()
    print(f"Video saved as {output_video}")


# --- CONFIGURATION ---
 
image_folder = "final_output/images_20250404111435"
voice_generates_folder = "final_output/voices_20250404111435"
output_path = "final_output/armanresult57.mp4"
fps = 1
voices_path = load_voices(voice_generates_folder)
image_paths = load_images(image_folder)
 

display_durations = []
for file_path in voices_path:
    try:
        audio = AudioFileClip(file_path)
        duration_sec = audio.duration
        display_durations.append(duration_sec)
        print(f"Duration of {file_path}: {duration_sec} seconds")
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

 
images_to_video_with_effects(
    image_paths, output_video=output_path, fps=fps, durations=display_durations
)

voices_root_folder:final_output/voices_20250404111435
Duration of final_output/voices_20250404111435\voice_scene_1.mp3: 16.85 seconds
Duration of final_output/voices_20250404111435\voice_scene_2.mp3: 7.85 seconds
Duration of final_output/voices_20250404111435\voice_scene_3.mp3: 10.51 seconds
Duration of final_output/voices_20250404111435\voice_scene_4.mp3: 7.92 seconds
Duration of final_output/voices_20250404111435\voice_scene_5.mp3: 14.4 seconds
Total frames: 16
Total frames written: 16
Total frames: 7
Total frames written: 7
Total frames: 10
Total frames written: 11
Total frames: 7
Total frames written: 7
Total frames: 14
Total frames written: 15
Video saved as final_output/armanresult57.mp4


In [19]:
import cv2
import numpy as np
import os
import math
from moviepy  import *


# === EFFECT FUNCTIONS ===

def zoom_in(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-in effect."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + (i / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames

def fade_out(image, num_frames=30):
    """Creates a fade-out effect from image to black."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = 1 - i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames

# === VIDEO GENERATION FUNCTION ===

def image_to_video_with_effects(image_path, audio_path, output_video, fps=30, fade_out_sec=2): 
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot load image: {image_path}")

    # Get image size
    h, w, _ = img.shape
    
    # Extract audio duration (only for timing)
    audio = AudioFileClip(audio_path)
    audio_duration = audio.duration
    total_frames = math.ceil(audio_duration * fps)
    
    # Compute fade-out duration in frames
    fade_out_frames = min(fps * fade_out_sec, total_frames // 2)  # Ensuring it's not too long
    main_effect_frames = total_frames - fade_out_frames  # Remaining frames for zoom-in

    print(f"🎵 Audio Duration (For Reference): {audio_duration:.2f} sec")
    print(f"🎬 Expected Video Duration: {total_frames / fps:.2f} sec")
    print(f"📌 Zoom-in Frames: {main_effect_frames}, Fade-out Frames: {fade_out_frames}")

    # Apply Zoom-in and Fade-out effects
    zoom_in_frames = zoom_in(img, num_frames=main_effect_frames, zoom_factor=0.5)
    fade_out_frames_list = fade_out(zoom_in_frames[-1], num_frames=int(fade_out_frames))
    
    # Combine frames
    effect_frames = zoom_in_frames + fade_out_frames_list
    
    # Create Video Writer
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video_writer = cv2.VideoWriter(output_video, fourcc, fps, (w, h))

    # Write frames
    for frame in effect_frames:
        video_writer.write(frame)
    
    # Release writer
    video_writer.release()

    # Load final video to check duration
    final_video = VideoFileClip(output_video)
    final_video_duration = final_video.duration
    print(f"✅ Final Video Duration: {final_video_duration:.2f} sec (No Audio)")

# === CONFIGURATION ===
image_path = "final_output/images_20250404111435/image_scene_1.png"
audio_path = "final_output/voices_20250404111435/voice_scene_1.mp3"  # Example audio file
output_path = "output_video_no_audio.mp4"
fps = 30
fade_out_duration = 2  # Fade-out duration in seconds

# Generate Video (Without Audio)
image_to_video_with_effects(image_path, audio_path, output_path, fps, fade_out_duration)

🎵 Audio Duration (For Reference): 16.85 sec
🎬 Expected Video Duration: 16.87 sec
📌 Zoom-in Frames: 446, Fade-out Frames: 60
{'video_found': True, 'audio_found': False, 'metadata': {'major_brand': 'isom', 'minor_version': '512', 'compatible_brands': 'isomiso2mp41', 'encoder': 'Lavf58.76.100'}, 'inputs': [{'streams': [{'input_number': 0, 'stream_number': 0, 'stream_type': 'video', 'language': None, 'default': True, 'size': [1080, 1920], 'bitrate': 2201, 'fps': 30.0, 'codec_name': 'mpeg4', 'profile': '(Simple Profile)', 'metadata': {'handler_name': 'VideoHandler'}}], 'input_number': 0}], 'duration': 16.87, 'bitrate': 2203, 'start': 0.0, 'default_video_input_number': 0, 'default_video_stream_number': 0, 'video_codec_name': 'mpeg4', 'video_profile': '(Simple Profile)', 'video_size': [1080, 1920], 'video_bitrate': 2201, 'video_fps': 30.0, 'video_duration': 16.87, 'video_n_frames': 506}
c:\Users\arman\AppData\Local\Programs\Python\Python312\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win